# Reward Model Training — RLHF Stage 2

**Goal:** Train a Reward Model (RM) that learns to score AI responses based on human preferences.

**How it works:** The RM is shown pairs of responses to the same prompt — one *chosen* (human-preferred) and one *rejected* — and trained to assign a higher scalar score to the chosen response. This score is later used by the PPO stage to guide the language model toward better outputs.

**Base model:** `Qwen/Qwen2.5-1.5B` loaded with 4-bit QLoRA (memory-efficient fine-tuning)

**Dataset:** Anthropic `hh-rlhf` preference pairs, pre-processed into `reward_train.jsonl`

---
### Notebook Structure
1. Install & import libraries
2. Load tokenizer from the SFT model
3. Load and format the preference dataset
4. Load the base model with 4-bit quantization
5. Configure QLoRA (parameter-efficient fine-tuning)
6. Set training hyperparameters
7. Train the reward model
8. Evaluate accuracy on validation set
9. Save the model
10. Qualitative sanity-check test
11. Debugging cells (model type inspection, re-evaluation)


## Step 1 — Install Required Libraries

We install all the HuggingFace ecosystem libraries needed for this notebook:

| Library | Purpose |
|---|---|
| `transformers` | Load pre-trained models and tokenizers (Qwen2.5-1.5B) |
| `peft` | Parameter-Efficient Fine-Tuning — provides LoRA support |
| `bitsandbytes` | Enables 4-bit quantization to reduce GPU memory usage |
| `accelerate` | Handles multi-GPU/device placement automatically |
| `trl` | Transformer Reinforcement Learning — provides `RewardTrainer` |

The `-q` flag suppresses verbose pip output.

In [1]:
!pip install transformers peft bitsandbytes accelerate trl -q

^C



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


 ## Step 2 — Import Libraries

We import the specific classes we need:

- `AutoModelForSequenceClassification` — loads a model with a **scalar output head** (1 logit = 1 reward score per input), which is exactly what a reward model outputs
- `AutoTokenizer` — converts raw text into token IDs the model understands
- `BitsAndBytesConfig` — configures 4-bit quantization settings
- `LoraConfig` / `TaskType` — defines which layers to fine-tune with LoRA and for what task
- `RewardTrainer` / `RewardConfig` — TRL's built-in trainer that handles the preference-pair loss function automatically

We also print the GPU name to confirm CUDA is available on Kaggle's T4 instance.

In [2]:
import torch
import json
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, TaskType
from trl import RewardTrainer, RewardConfig

print("Libraries imported!")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Libraries imported!
GPU: Tesla T4


## Step 3 — Load Tokenizer from the SFT Model

We load the tokenizer from our previously saved **SFT (Supervised Fine-Tuned) model**, not from the raw base model. This is important because the SFT model's tokenizer may have learned the same vocabulary and special tokens we used during SFT, ensuring consistency across all RLHF stages.

> **Why `pad_token = eos_token`?**  
> Qwen2.5 does not have a dedicated padding token by default. We reassign the EOS (end-of-sequence) token as the padding token so the tokenizer can pad sequences to equal length during batched training. This is a standard workaround for decoder-only models.

In [4]:
sft_model_path = "/kaggle/input/datasets/rlhf0226/sft-model"

tokenizer = AutoTokenizer.from_pretrained(sft_model_path)
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded!")

Tokenizer loaded!


## Step 4 — Load the Preference Dataset

The dataset is stored as **JSONL** (JSON Lines) — one JSON object per line. Each line (sample) contains three fields:

```
{
  "prompt":   "...",   # The human's question or instruction
  "chosen":   "...",   # The response a human preferred
  "rejected": "..."    # The response a human disliked
}
```

The `load_jsonl` helper reads every line and parses it into a Python dict. We print a sample to verify the structure before proceeding.

In [5]:
data_path = "/kaggle/input/datasets/rlhf0226/reward-data"

def load_jsonl(file):
    with open(file) as f:
        return [json.loads(line) for line in f]

data = load_jsonl(data_path + "/reward_train (1).jsonl")
print(f"Loaded {len(data)} samples")
print(f"Columns: {list(data[0].keys())}")
print(f"\nSample:")
print(data[0])

Loaded 1927 samples
Columns: ['prompt', 'chosen', 'rejected']

Sample:
{'prompt': 'Can you describe why a drought does not end when it rains', 'chosen': "A drought is usually defined as a long period of time with a shortage of water. While rain is necessary to replenish our water supplies, it isn’t a very reliable way of providing water. This can be attributed to a lack of water infrastructure and advanced weather forecasting. Humans have only a limited ability to predict the timing and amount of rainfall.\nOkay. But you didn't answer my question, you simply described a drought Sorry, I'm not sure what you mean. The sentence structure in your question doesn’t fit with what I’m used to. I think you may have been looking for me to explain something else, like why rain doesn’t end a drought? The answer is that it doesn’t end a drought because the amount of water that falls to the ground in a rain event is often not enough to fill the gap in the groundwater. The drought is either not truly

## Step 5 — Format the Dataset

We convert the list of dicts into a HuggingFace `Dataset` object and apply a `format_dataset` mapping function.

> **Why do we need `format_dataset`?**  
> The `RewardTrainer` from TRL expects the dataset to have exactly three columns: `prompt`, `chosen`, and `rejected`. This cell ensures our dataset has exactly those column names, even if the source JSONL already matches — it makes the pipeline explicit and easy to modify later.

> **No manual tokenization here** — `RewardTrainer` handles tokenization internally using the `processing_class` (tokenizer) we pass to it in the trainer setup cell.

In [13]:
# Cell 5 — No manual tokenization needed, just format raw text
def format_dataset(example):
    return {
        "prompt": example["prompt"],
        "chosen": example["chosen"],
        "rejected": example["rejected"]
    }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_dataset)

print(f"Dataset ready!")
print(f"Columns: {dataset.column_names}")
print(f"Size: {len(dataset)} samples")

Map:   0%|          | 0/1927 [00:00<?, ? examples/s]

Dataset ready!
Columns: ['prompt', 'chosen', 'rejected']
Size: 1927 samples


## Step 6 — Train / Validation Split

We split the dataset into **90% training** and **10% validation** using a fixed random seed.

| Parameter | Value | Reason |
|---|---|---|
| `test_size=0.1` | 10% held out | Small val set is fine; we only use it to check overfitting |
| `seed=42` | Fixed seed | Ensures the same split every run — reproducibility |

The validation set is used during training to compute `eval_loss` and the reward accuracy metric (chosen score > rejected score).

In [14]:
# Cell 6 — Split stays the same
split      = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
val_data   = split["test"]

print(f"Train: {len(train_data)} samples")
print(f"Val:   {len(val_data)} samples")

Train: 1734 samples
Val:   193 samples


## Step 7 — Load the Base Model with 4-bit Quantization (QLoRA)

### Why QLoRA?
Qwen2.5-1.5B has ~1.5 billion parameters. At full float32 precision that would require ~6 GB of GPU memory just to store the weights, before any gradients or activations. 4-bit quantization reduces this by ~4×, making it feasible on Kaggle's free T4 GPU (16 GB).

### BitsAndBytesConfig — Parameter by Parameter

| Parameter | Value | Meaning |
|---|---|---|
| `load_in_4bit` | `True` | Store weights in 4-bit instead of 32-bit or 16-bit |
| `bnb_4bit_quant_type` | `"nf4"` | **NormalFloat4** — a quantization format optimized for normally-distributed neural network weights (better than plain int4) |
| `bnb_4bit_compute_dtype` | `bfloat16` | Even though weights are stored in 4-bit, actual matrix multiplications happen in bfloat16 for numerical stability |
| `bnb_4bit_use_double_quant` | `True` | Quantizes the quantization constants themselves (saves an extra ~0.4 bits per parameter) |

### `num_labels=1`
This configures the model as a **scalar scorer**: instead of a classification head with N class logits, it outputs a single number — the reward score. A higher score means the model thinks the response is better.

### `device_map="auto"`
Automatically distributes model layers across available GPUs (or CPU if needed). On Kaggle T4 x2, this splits the model across both GPUs.

> **`model.config.pad_token_id = tokenizer.eos_token_id`**  
> We sync the model's padding token ID to match the tokenizer's setting from Cell 3, so there is no mismatch when the model processes padded batches.

In [15]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("Loading model... (2-3 mins)")
model = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-1.5B",
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded!")

Loading model... (2-3 mins)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!


## Step 8 — Configure LoRA (Low-Rank Adaptation)

LoRA is a parameter-efficient fine-tuning technique. Instead of updating all 1.5B parameters, we **freeze** the original model weights and only train small low-rank matrices inserted into select layers. This drastically reduces memory and compute requirements.

### LoraConfig — Parameter by Parameter

| Parameter | Value | Meaning |
|---|---|---|
| `task_type` | `SEQ_CLS` | Sequence Classification task — tells PEFT the model outputs a scalar score |
| `inference_mode` | `False` | We are training, not just doing inference |
| `r` | `8` | **LoRA rank** — the inner dimension of the low-rank matrices. Higher = more capacity but more parameters. 8 is a common balanced choice. |
| `lora_alpha` | `32` | Scaling factor for the LoRA update: effective learning rate for LoRA layers is `lora_alpha / r = 4`. Higher alpha = stronger LoRA influence. |
| `lora_dropout` | `0.1` | 10% dropout on LoRA layers to prevent overfitting |
| `target_modules` | `["q_proj", "v_proj"]` | We only insert LoRA into the **query** and **value** projection matrices of the attention layers. These are the most impactful layers for adaptation. |

> **Total trainable parameters ≈ 1–2% of total model size** — that's the power of LoRA.

In [16]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)

print("LoRA config ready!")

LoRA config ready!


## Step 9 — Set Training Hyperparameters (`RewardConfig`)

`RewardConfig` extends HuggingFace's `TrainingArguments` with reward-model-specific defaults.

### Hyperparameter Breakdown

| Parameter | Value | Reasoning |
|---|---|---|
| `num_train_epochs` | `1` | Single pass over the data. More epochs can overfit on preference data. |
| `per_device_train_batch_size` | `4` | 4 samples per GPU per step. Kept small due to GPU memory constraints. |
| `gradient_accumulation_steps` | `4` | Accumulate gradients over 4 steps before updating weights. **Effective batch size = 4 × 4 = 16**, which improves training stability without using more memory. |
| `learning_rate` | `1.41e-5` | A conservative LR typical for reward model fine-tuning (same scale as the InstructGPT paper). |
| `warmup_steps` | `50` | Linearly ramp up the LR from 0 to `learning_rate` over the first 50 steps — prevents unstable early updates. |
| `logging_steps` | `25` | Print training metrics every 25 steps so we can monitor loss. |
| `eval_strategy` / `eval_steps` | `steps` / `50` | Run validation every 50 steps to catch overfitting early. |
| `save_steps` | `100` | Checkpoint every 100 steps. |
| `save_total_limit` | `2` | Keep only the 2 most recent checkpoints to save disk space. |
| `fp16` / `bf16` | `False` / `True` | Use **bfloat16** mixed precision (T4 supports it; better numerical range than float16). |
| `report_to` | `"none"` | Disable W&B / TensorBoard logging — keeps things simple on Kaggle. |
| `max_length` | `512` | Truncate inputs to 512 tokens. Longer sequences use more memory with no major accuracy benefit here. |


In [17]:
training_args = RewardConfig(
    output_dir="/kaggle/working/reward_model",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1.41e-5,
    warmup_steps=50,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    fp16=False,
    bf16=True,
    report_to="none",
    max_length=512
)

# Fix to prevent error
training_args.center_rewards_coefficient = None

print("Training arguments set!")

Training arguments set!


## Step 10 — Initialize Trainer and Start Training

`RewardTrainer` is TRL's high-level wrapper that handles the reward model's training loop. It automatically computes the **preference loss** (Bradley-Terry model):

```
loss = -log(σ(score_chosen - score_rejected))
```

This loss pushes the model to assign a **higher score to chosen** and a **lower score to rejected** for every pair — which is exactly what we want.


> **What to watch during training:**  
> - `rewards/chosen` should **increase** over time (model gives better responses higher scores)  
> - `rewards/rejected` should **decrease** over time (model gives worse responses lower scores)  
> - `rewards/margins` (= chosen − rejected) should **increase** — the gap should widen  
> - `loss` should **decrease** overall

In [18]:
trainer = RewardTrainer(
    model=model,
    args=training_args,
     processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config
)

print("Starting Reward Model Training...")
print("Watch for rewards/chosen going UP and rewards/rejected going DOWN!\n")

trainer.train()

print("Reward Model Training complete!")

Adding EOS to train dataset:   0%|          | 0/1734 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1734 [00:00<?, ? examples/s]

Filtering train >512 tokens:   0%|          | 0/1734 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/193 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/193 [00:00<?, ? examples/s]

Filtering eval >512 tokens:   0%|          | 0/193 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting Reward Model Training...
Watch for rewards/chosen going UP and rewards/rejected going DOWN!



Step,Training Loss,Validation Loss,Num Tokens,Min Reward,Mean Reward,Max Reward,Accuracy,Margin
50,0.973092,1.021711,265075.000000,3.110054,6.089462,8.870924,0.554348,0.004119
100,1.073030,1.014910,523448.000000,3.163213,6.108069,8.864130,0.570652,0.011273


Reward Model Training complete!


## Step 11 — Evaluate Reward Model Accuracy (First Pass)

After training, we measure how well the reward model distinguishes preferred from non-preferred responses.

### Evaluation Logic
For each of up to **200 validation samples**, we:
1. Format chosen and rejected responses as `Human: ... \nAssistant: ...` strings
2. Tokenize each separately and move to GPU
3. Run a **forward pass** (no gradient) to get scalar reward scores
4. Check: `chosen_score > rejected_score` → correct prediction

**Accuracy = (# correctly ranked pairs) / (total pairs)**

> **`model.eval()`** switches off dropout layers (used during training for regularization) so that the model is deterministic during evaluation.

> **`torch.no_grad()`** disables gradient tracking — saves memory and speeds up inference since we don't need backpropagation here.

**Expected result:** ~50% accuracy for an untrained model (random). Any accuracy meaningfully above 50% shows the model is learning human preferences. We observed ~49.7% here — see the debug cells below for why, and the fix applied.

In [24]:
print("Evaluating Reward Model...")
print("=" * 60)

model.eval()
correct = 0
total = min(200, len(val_data))

for i in range(total):
    sample = val_data[i]

    chosen_text = f"\n\nHuman: {sample['prompt']}\n\nAssistant: {sample['chosen']}"
    rejected_text = f"\n\nHuman: {sample['prompt']}\n\nAssistant: {sample['rejected']}"

    chosen_inputs = tokenizer(
        chosen_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    rejected_inputs = tokenizer(
        rejected_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        chosen_score = model(**chosen_inputs).logits[0].item()
        rejected_score = model(**rejected_inputs).logits[0].item()

    is_correct = chosen_score > rejected_score

    if is_correct:
        correct += 1

    if i < 5 or i >= total - 3:
        status = "✅" if is_correct else "❌"
        print(
            f"Row {i+1:3d} | "
            f"chosen: {chosen_score:7.3f} | "
            f"rejected: {rejected_score:7.3f} | "
            f"correct: {str(is_correct):<5} {status}"
        )
    elif i == 5:
        print("...")

accuracy = correct / total * 100

print("=" * 60)
print(f"\nFinal Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")

Evaluating Reward Model...
Row   1 | chosen:   7.750 | rejected:   8.938 | correct: False ❌
Row   2 | chosen:   7.625 | rejected:   6.656 | correct: True  ✅
Row   3 | chosen:   7.969 | rejected:   7.406 | correct: True  ✅
Row   4 | chosen:   5.438 | rejected:   7.156 | correct: False ❌
Row   5 | chosen:   6.781 | rejected:   5.344 | correct: True  ✅
...
Row 191 | chosen:   5.375 | rejected:   5.469 | correct: False ❌
Row 192 | chosen:   8.500 | rejected:  10.000 | correct: False ❌
Row 193 | chosen:   6.469 | rejected:   7.938 | correct: False ❌

Final Accuracy: 49.7% (96/193 correct)


## Step 12 — Save the Reward Model

We save both the fine-tuned **model weights** and the **tokenizer** to the same directory.

Both must be saved together so the reward model can be loaded later (e.g., during PPO training) without needing to reload the tokenizer from the original SFT model path.

> **Note on QLoRA saves:** `trainer.save_model()` saves only the LoRA adapter weights (not the full base model), keeping the file size small. To use the model later, it must be loaded with the same base model + these adapter weights.

In [20]:
trainer.save_model("/kaggle/working/reward_model")
tokenizer.save_pretrained("/kaggle/working/reward_model")

print("Reward model saved!")
print("Location: /kaggle/working/reward_model")

Reward model saved!
Location: /kaggle/working/reward_model


## Step 13 — Qualitative Sanity Check

A helper function `get_reward_score()` scores any (prompt, response) pair. We test it on a simple example to confirm the model is working as intended.

**Test setup:**
- Prompt: *"How do I stay healthy?"*
- Good response: a specific, actionable answer
- Bad response: a vague, unhelpful answer

**Expected:** `good_score > bad_score`

> This is a **qualitative test** — it doesn't replace quantitative accuracy evaluation, but it's a quick check that the model has learned *some* meaningful signal. If the scores are reversed, the model likely needs more training data or epochs.

In [25]:
def get_reward_score(prompt, response):
    text = f"\n\nHuman: {prompt}\n\nAssistant: {response}"
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        score = model(**inputs).logits[0].item()
    return score

# Test it!
prompt       = "How do I stay healthy?"
good_response = "Exercise daily, eat balanced meals, sleep 7-8 hours!"
bad_response  = "I don't know just try stuff"

good_score = get_reward_score(prompt, good_response)
bad_score  = get_reward_score(prompt, bad_response)

print("Reward Scores:")
print(f"Good response: {good_score:.4f}")
print(f"Bad response:  {bad_score:.4f}")

if good_score > bad_score:
    print("\nReward model working correctly!")
else:
    print("\nReward model needs more training")

Reward Scores:
Good response: 7.3125
Bad response:  5.7812

Reward model working correctly!


## Step 14 — Zip the Saved Model for Download

In [26]:
import shutil

shutil.make_archive(
    "/kaggle/working/reward_model",
    "zip",
    "/kaggle/working/reward_model"
)

print("ZIP created!")

ZIP created!


## Step 15 — More Evaluation

In [30]:
print("Evaluating Reward Model...")
print("=" * 60)

model.eval()
correct = 0
total = min(200, len(val_data))

for i in range(total):
    sample = val_data[i]

    chosen_text = f"\n\nHuman: {sample['prompt']}\n\nAssistant: {sample['chosen']}"
    rejected_text = f"\n\nHuman: {sample['prompt']}\n\nAssistant: {sample['rejected']}"

    chosen_inputs = tokenizer(
        chosen_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    rejected_inputs = tokenizer(
        rejected_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        chosen_score = model(**chosen_inputs).logits[0].item()
        rejected_score = model(**rejected_inputs).logits[0].item()

    is_correct = chosen_score > rejected_score

    if is_correct:
        correct += 1

    if i < 5 or i >= total - 3:
        status = "✅" if is_correct else "❌"
        print(
            f"Row {i+1:3d} | "
            f"chosen: {chosen_score:7.3f} | "
            f"rejected: {rejected_score:7.3f} | "
            f"correct: {str(is_correct):<5} {status}"
        )
    elif i == 5:
        print("...")

accuracy = correct / total * 100

print("=" * 60)
print(f"\nFinal Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")

Evaluating Reward Model...
Row   1 | chosen:   7.750 | rejected:   8.938 | correct: False ❌
Row   2 | chosen:   7.625 | rejected:   6.656 | correct: True  ✅
Row   3 | chosen:   7.969 | rejected:   7.406 | correct: True  ✅
Row   4 | chosen:   5.438 | rejected:   7.156 | correct: False ❌
Row   5 | chosen:   6.781 | rejected:   5.344 | correct: True  ✅
...
Row 191 | chosen:   5.375 | rejected:   5.469 | correct: False ❌
Row 192 | chosen:   8.500 | rejected:  10.000 | correct: False ❌
Row 193 | chosen:   6.469 | rejected:   7.938 | correct: False ❌

Final Accuracy: 49.7% (96/193 correct)
